In [1]:
!pip -q install python-dotenv

In [2]:
from getpass import getpass

api_key = getpass("Enter your Groq API key: ")

with open(".env", "w") as f:
    f.write(f"GROQ_API_KEY={api_key}\n")

print("API key saved securely to .env")

Enter your Groq API key: ··········
API key saved securely to .env


In [3]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY was not found in .env")

client = Groq(api_key=api_key)

print("Groq API key loaded successfully from .env.")

Groq API key loaded successfully from .env.


In [4]:
print("API key exists:", bool(api_key))
print("Actual API key value is not displayed.")

API key exists: True
Actual API key value is not displayed.


In [5]:
# Task 1: THREE PROMPT TEMPLATES

import pandas as pd
import os

# Load the clothing review dataset

csv_path = "/content/Womens Clothing E-Commerce Reviews.csv"

df = pd.read_csv(csv_path)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))


# Get one real customer review

review = df["Review Text"].dropna().iloc[0]

print("\nExample review:")
print(review)


# a :

A_ZERO_SHOT_PROMPT = """
You are performing sentiment classification on a customer clothing review.

Classify the overall sentiment of the review as positive, negative, or neutral.

Return ONLY valid JSON using exactly this schema:

{{
  "label": "positive|negative|neutral",
  "confidence": "low|medium|high",
  "reason": "string"
}}

Do not add any other fields.
Do not use Markdown.
Do not wrap the JSON in code fences.

Customer review:
{review}
"""


# b :

B_FEW_SHOT_PROMPT = """
You are performing sentiment classification on a customer clothing review.

Classify the overall sentiment as positive, negative, or neutral.

Use the following examples as guidance.

Example 1:
Review: "Absolutely love this dress. The fabric is beautiful and it fits perfectly."
Output:
{{"label":"positive","confidence":"high","reason":"The customer expresses strong satisfaction with the dress, fabric, and fit."}}

Example 2:
Review: "The material feels cheap and the sizing is completely wrong."
Output:
{{"label":"negative","confidence":"high","reason":"The customer is dissatisfied with both the material and sizing."}}

Example 3:
Review: "The product is okay. It is not especially good or bad."
Output:
{{"label":"neutral","confidence":"medium","reason":"The customer gives a balanced opinion without clearly positive or negative sentiment."}}

Return ONLY valid JSON using exactly this schema:

{{
  "label": "positive|negative|neutral",
  "confidence": "low|medium|high",
  "reason": "string"
}}

Do not add any other fields.
Do not use Markdown.
Do not wrap the JSON in code fences.

Customer review:
{review}
"""

# c :

C_ROLE_PROMPTED_PROMPT = """
Act as a senior customer-insights analyst specializing in
e-commerce clothing reviews.

Your task is to determine the overall sentiment of the
customer review.

Use the Three Cs framework:

1. Clarity:
   Clearly identify the overall sentiment.

2. Context:
   Consider the customer's comments about the clothing,
   including quality, comfort, appearance, sizing, fit,
   material, or overall satisfaction.

3. Constraint:
   Return ONLY the required JSON structure and do not
   include additional fields or commentary.

Return ONLY valid JSON using exactly this schema:

{{
  "label": "positive|negative|neutral",
  "confidence": "low|medium|high",
  "reason": "string"
}}

Do not add any other fields.
Do not use Markdown.
Do not wrap the JSON in code fences.

Customer review:
{review}
"""

# DISPLAY ALL THREE TASK 1 PROMPTS

print("\n\n==============================")
print("TASK 1A — ZERO-SHOT")
print("==============================")
print(A_ZERO_SHOT_PROMPT.format(review=review))

print("\n\n==============================")
print("TASK 1B — FEW-SHOT")
print("==============================")
print(B_FEW_SHOT_PROMPT.format(review=review))

print("\n\n==============================")
print("TASK 1C — ROLE-PROMPTED")
print("==============================")
print(C_ROLE_PROMPTED_PROMPT.format(review=review))


print("\n\n======================================")
print("TASK 1 COMPLETE")
print("======================================")

Dataset loaded successfully.
Rows: 23486
Columns: 11

Example review:
Absolutely wonderful - silky and sexy and comfortable


TASK 1A — ZERO-SHOT

You are performing sentiment classification on a customer clothing review.

Classify the overall sentiment of the review as positive, negative, or neutral.

Return ONLY valid JSON using exactly this schema:

{
  "label": "positive|negative|neutral",
  "confidence": "low|medium|high",
  "reason": "string"
}

Do not add any other fields.
Do not use Markdown.
Do not wrap the JSON in code fences.

Customer review:
Absolutely wonderful - silky and sexy and comfortable



TASK 1B — FEW-SHOT

You are performing sentiment classification on a customer clothing review.

Classify the overall sentiment as positive, negative, or neutral.

Use the following examples as guidance.

Example 1:
Review: "Absolutely love this dress. The fabric is beautiful and it fits perfectly."
Output:
{"label":"positive","confidence":"high","reason":"The customer expresses s

In [6]:
!pip install groq

In [7]:
# Task 2: REUSABLE GROQ API WRAPPER

# Install Groq if needed
!pip -q install groq

# Import libraries
from google.colab import userdata
from groq import Groq

# Load Groq API key from Colab Secret
# Secret name must be: GROQ_API_KEY

api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY was not found. "
        "Make sure the Colab Secret is named GROQ_API_KEY "
        "and Notebook access is ON."
    )

print("Groq API key loaded successfully.")

# Create Groq client

client = Groq(api_key=api_key)

# Reusable LLM API function

def call_llm(prompt, temperature, max_tokens):
    """
    Sends a prompt to the Groq LLM API.

    Parameters:
        prompt       : Prompt text to send to the model
        temperature  : Controls response randomness
        max_tokens   : Maximum number of output tokens

    Returns:
        The model's text response.
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )

    return response.choices[0].message.content


# TEST THE REUSABLE API WRAPPER

# Get one real review from the dataset
test_review = df["Review Text"].dropna().iloc[0]

# Use the Task 1A zero-shot prompt
test_prompt = A_ZERO_SHOT_PROMPT.format(
    review=test_review
)

# Call the reusable function
test_response = call_llm(
    prompt=test_prompt,
    temperature=0.0,
    max_tokens=200
)

# Display result

print("\n==============================")
print("PART 3 — TASK 2")
print("REUSABLE GROQ API WRAPPER")
print("==============================")

print("\nReview:")
print(test_review)

print("\nModel response:")
print(test_response)

print("\n==============================")
print("TASK 2 COMPLETE")
print("==============================")

Groq API key loaded successfully.

PART 3 — TASK 2
REUSABLE GROQ API WRAPPER

Review:
Absolutely wonderful - silky and sexy and comfortable

Model response:
{
  "label": "positive",
  "confidence": "high",
  "reason": "The review contains positive adjectives such as 'wonderful', 'silky', 'sexy', and 'comfortable'."
}

TASK 2 COMPLETE


In [8]:
# Task 3: RETRY-ON-FAILURE HANDLING

import time

def call_llm_with_retry(prompt, temperature, max_tokens):
    """
    Calls the Groq API with retry handling.

    The function retries up to 3 times if an API error occurs.
    If all attempts fail, it prints an error and returns None.
    """

    max_retries = 3

    for attempt in range(1, max_retries + 1):

        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )

            return response.choices[0].message.content

        except Exception as error:

            print(f"Attempt {attempt} failed: {error}")

            if attempt < max_retries:
                print("Retrying...")
                time.sleep(2)
            else:
                print("All 3 attempts failed.")
                print("Moving on without crashing the program.")
                return None


# TEST TASK 3

test_review = df["Review Text"].dropna().iloc[0]

test_prompt = A_ZERO_SHOT_PROMPT.format(
    review=test_review
)

test_response = call_llm_with_retry(
    prompt=test_prompt,
    temperature=0.0,
    max_tokens=200
)


# DISPLAY RESULT

print("\n==============================")
print("PART 3 — TASK 3")
print("RETRY-ON-FAILURE HANDLING")
print("==============================")

print("\nReview:")
print(test_review)

print("\nModel response:")
print(test_response)

print("\n==============================")
print("TASK 3 COMPLETE")
print("==============================")


PART 3 — TASK 3
RETRY-ON-FAILURE HANDLING

Review:
Absolutely wonderful - silky and sexy and comfortable

Model response:
{
  "label": "positive",
  "confidence": "high",
  "reason": "The review contains positive adjectives such as 'wonderful', 'silky', 'sexy', and 'comfortable'."
}

TASK 3 COMPLETE


In [9]:
# TAsk 4: PROMPT COMPARISON
# 3 PROMPTS × 5 REAL REVIEWS = 15 API CALLS

import json
import pandas as pd

# Select 5 real customer reviews

reviews = df["Review Text"].dropna().head(5).tolist()

print("Number of reviews selected:", len(reviews))


# Store the three prompt templates

prompts = {
    "A — Zero-shot": A_ZERO_SHOT_PROMPT,
    "B — Few-shot": B_FEW_SHOT_PROMPT,
    "C — Role-prompted": C_ROLE_PROMPTED_PROMPT
}


# Required JSON fields

required_fields = {
    "label",
    "confidence",
    "reason"
}


# Run all 15 API calls

results = []

for template_name, template in prompts.items():

    for i, review in enumerate(reviews, start=1):

        print(f"Running {template_name} — Review {i}/5")

        prompt = template.format(review=review)

        response = call_llm_with_retry(
            prompt=prompt,
            temperature=0.0,
            max_tokens=200
        )

        json_valid = False
        schema_valid = False
        parsed_response = None

        if response is not None:

            try:
                parsed_response = json.loads(response)
                json_valid = True

                if (
                    isinstance(parsed_response, dict)
                    and set(parsed_response.keys()) == required_fields
                    and parsed_response["label"] in [
                        "positive",
                        "negative",
                        "neutral"
                    ]
                    and parsed_response["confidence"] in [
                        "low",
                        "medium",
                        "high"
                    ]
                    and isinstance(parsed_response["reason"], str)
                ):
                    schema_valid = True

            except json.JSONDecodeError:
                json_valid = False

        results.append({
            "template": template_name,
            "review_number": i,
            "review": review,
            "raw_response": response,
            "json_valid": json_valid,
            "schema_valid": schema_valid
        })


# Create results DataFrame

results_df = pd.DataFrame(results)


# Task 4 — RESULTS

print("\n======================================")
print("PART 3 — TASK 4 RESULTS")
print("======================================")

print("\nTotal API calls:", len(results_df))

print("\nDetailed results:")
display(
    results_df[
        [
            "template",
            "review_number",
            "json_valid",
            "schema_valid"
        ]
    ]
)


# Compare templates

comparison = (
    results_df
    .groupby("template")
    .agg(
        total_calls=("template", "count"),
        valid_json=("json_valid", "sum"),
        valid_schema=("schema_valid", "sum")
    )
    .reset_index()
)

comparison["JSON_success_rate"] = (
    comparison["valid_json"] / comparison["total_calls"] * 100
)

comparison["schema_success_rate"] = (
    comparison["valid_schema"] / comparison["total_calls"] * 100
)


print("\n======================================")
print("PROMPT COMPARISON")
print("======================================")

display(comparison)


# Identify the most consistent template

best_template = comparison.sort_values(
    ["schema_success_rate", "JSON_success_rate"],
    ascending=False
).iloc[0]["template"]

print("\nMost consistently schema-conformant template:")
print(best_template)

print("\n======================================")
print("TASK 4 COMPLETE")
print("======================================")

Number of reviews selected: 5
Running A — Zero-shot — Review 1/5
Running A — Zero-shot — Review 2/5
Running A — Zero-shot — Review 3/5
Running A — Zero-shot — Review 4/5
Running A — Zero-shot — Review 5/5
Running B — Few-shot — Review 1/5
Running B — Few-shot — Review 2/5
Running B — Few-shot — Review 3/5
Running B — Few-shot — Review 4/5
Running B — Few-shot — Review 5/5
Running C — Role-prompted — Review 1/5
Running C — Role-prompted — Review 2/5
Running C — Role-prompted — Review 3/5
Running C — Role-prompted — Review 4/5
Running C — Role-prompted — Review 5/5

PART 3 — TASK 4 RESULTS

Total API calls: 15

Detailed results:


,template,review_number,json_valid,schema_valid
0,A — Zero-shot,1,True,True
1,A — Zero-shot,2,True,True
2,A — Zero-shot,3,True,True
3,A — Zero-shot,4,True,True
4,A — Zero-shot,5,True,True
5,B — Few-shot,1,True,True
6,B — Few-shot,2,True,True
7,B — Few-shot,3,True,True
8,B — Few-shot,4,True,True
9,B — Few-shot,5,True,True



PROMPT COMPARISON


,template,total_calls,valid_json,valid_schema,JSON_success_rate,schema_success_rate
0,A — Zero-shot,5,5,5,100.0,100.0
1,B — Few-shot,5,5,5,100.0,100.0
2,C — Role-prompted,5,5,5,100.0,100.0



Most consistently schema-conformant template:
A — Zero-shot

TASK 4 COMPLETE


In [10]:
# Task 5: ASPECT-BASED SENTIMENT

import json
import pandas as pd

# Aspect-based prompt

ASPECT_PROMPT = """
You are performing aspect-based sentiment analysis on a
customer clothing review.

Identify the sentiment for each aspect mentioned in the review.

Use these aspects:
- quality
- comfort
- fit
- appearance
- material
- sizing

For each aspect, use:
positive, negative, neutral, or not_mentioned.

Return ONLY valid JSON using exactly this structure:

{{
  "quality": "positive|negative|neutral|not_mentioned",
  "comfort": "positive|negative|neutral|not_mentioned",
  "fit": "positive|negative|neutral|not_mentioned",
  "appearance": "positive|negative|neutral|not_mentioned",
  "material": "positive|negative|neutral|not_mentioned",
  "sizing": "positive|negative|neutral|not_mentioned"
}}

Do not add any other fields.
Do not use Markdown.
Do not wrap the JSON in code fences.

Customer review:
{review}
"""


# Select 5 real reviews

aspect_reviews = df["Review Text"].dropna().head(5).tolist()

aspect_results = []


# Run aspect-based analysis

for i, review in enumerate(aspect_reviews, start=1):

    print(f"Processing review {i}/5")

    prompt = ASPECT_PROMPT.format(review=review)

    response = call_llm_with_retry(
        prompt=prompt,
        temperature=0.0,
        max_tokens=200
    )

    parsed = None
    json_valid = False
    schema_valid = False

    if response is not None:

        try:
            parsed = json.loads(response)
            json_valid = True

            required_aspects = {
                "quality",
                "comfort",
                "fit",
                "appearance",
                "material",
                "sizing"
            }

            allowed_values = {
                "positive",
                "negative",
                "neutral",
                "not_mentioned"
            }

            if (
                isinstance(parsed, dict)
                and set(parsed.keys()) == required_aspects
                and all(
                    value in allowed_values
                    for value in parsed.values()
                )
            ):
                schema_valid = True

        except json.JSONDecodeError:
            pass

    aspect_results.append({
        "review_number": i,
        "review": review,
        "response": response,
        "json_valid": json_valid,
        "schema_valid": schema_valid,
        "quality": parsed.get("quality") if parsed else None,
        "comfort": parsed.get("comfort") if parsed else None,
        "fit": parsed.get("fit") if parsed else None,
        "appearance": parsed.get("appearance") if parsed else None,
        "material": parsed.get("material") if parsed else None,
        "sizing": parsed.get("sizing") if parsed else None
    })


# Create results table

aspect_df = pd.DataFrame(aspect_results)


# DISPLAY Task 5 RESULTS

print("\n======================================")
print("PART 3 — TASK 5")
print("ASPECT-BASED SENTIMENT")
print("======================================")

print("\nDetailed results:")

display(
    aspect_df[
        [
            "review_number",
            "quality",
            "comfort",
            "fit",
            "appearance",
            "material",
            "sizing",
            "json_valid",
            "schema_valid"
        ]
    ]
)


# Validation summary

print(
    "\nJSON responses:",
    aspect_df["json_valid"].sum(),
    "/",
    len(aspect_df)
)

print(
    "Schema-valid responses:",
    aspect_df["schema_valid"].sum(),
    "/",
    len(aspect_df)
)

print("\n======================================")
print("TASK 5 COMPLETE")
print("======================================")

Processing review 1/5
Processing review 2/5
Processing review 3/5
Processing review 4/5
Processing review 5/5

PART 3 — TASK 5
ASPECT-BASED SENTIMENT

Detailed results:


,review_number,quality,comfort,fit,appearance,material,sizing,json_valid,schema_valid
0,1,positive,positive,not_mentioned,positive,positive,not_mentioned,True,True
1,2,positive,not_mentioned,positive,positive,not_mentioned,negative,True,True
2,3,negative,positive,negative,neutral,negative,negative,True,True
3,4,positive,not_mentioned,not_mentioned,positive,not_mentioned,not_mentioned,True,True
4,5,positive,not_mentioned,positive,positive,not_mentioned,not_mentioned,True,True



JSON responses: 5 / 5
Schema-valid responses: 5 / 5

TASK 5 COMPLETE


In [11]:
# Task 6: RESPONSE DRAFTING

import json
import pandas as pd

# Response-drafting prompt

RESPONSE_PROMPT = """
You are a professional customer-service representative for
an e-commerce clothing company.

Read the customer's review and write an appropriate response.

The response should:
- Be polite and professional.
- Acknowledge the customer's experience.
- Be empathetic when the review is negative.
- Thank the customer when appropriate.
- Avoid making promises that are not supported by the review.
- Be concise and natural.

Return ONLY valid JSON using exactly this structure:

{{
  "sentiment": "positive|negative|neutral",
  "response": "string"
}}

Do not add any other fields.
Do not use Markdown.
Do not wrap the JSON in code fences.

Customer review:
{review}
"""


# Select 5 real reviews

response_reviews = df["Review Text"].dropna().head(5).tolist()

response_results = []


# Generate responses

for i, review in enumerate(response_reviews, start=1):

    print(f"Processing review {i}/5")

    prompt = RESPONSE_PROMPT.format(
        review=review
    )

    response = call_llm_with_retry(
        prompt=prompt,
        temperature=0.0,
        max_tokens=200
    )

    parsed = None
    json_valid = False
    schema_valid = False

    if response is not None:

        try:
            parsed = json.loads(response)
            json_valid = True

            required_fields = {
                "sentiment",
                "response"
            }

            allowed_sentiments = {
                "positive",
                "negative",
                "neutral"
            }

            if (
                isinstance(parsed, dict)
                and set(parsed.keys()) == required_fields
                and parsed["sentiment"] in allowed_sentiments
                and isinstance(parsed["response"], str)
                and len(parsed["response"].strip()) > 0
            ):
                schema_valid = True

        except json.JSONDecodeError:
            pass

    response_results.append({
        "review_number": i,
        "review": review,
        "sentiment": parsed.get("sentiment") if parsed else None,
        "drafted_response": parsed.get("response") if parsed else None,
        "json_valid": json_valid,
        "schema_valid": schema_valid
    })


# Create results DataFrame

response_df = pd.DataFrame(response_results)


# DISPLAY TASK 6 RESULTS

print("\n======================================")
print("PART 3 — TASK 6")
print("RESPONSE DRAFTING")
print("======================================")

display(
    response_df[
        [
            "review_number",
            "sentiment",
            "drafted_response",
            "json_valid",
            "schema_valid"
        ]
    ]
)


# Validation summary

print(
    "\nJSON responses:",
    response_df["json_valid"].sum(),
    "/",
    len(response_df)
)

print(
    "Schema-valid responses:",
    response_df["schema_valid"].sum(),
    "/",
    len(response_df)
)

print("\n======================================")
print("TASK 6 COMPLETE")
print("======================================")

Processing review 1/5
Processing review 2/5
Processing review 3/5
Processing review 4/5
Processing review 5/5

PART 3 — TASK 6
RESPONSE DRAFTING


,review_number,sentiment,drafted_response,json_valid,schema_valid
0,1,positive,Thank you so much for your kind words! We're t...,True,True
1,2,positive,Thank you so much for sharing your positive ex...,True,True
2,3,negative,Thank you for sharing your experience with us....,True,True
3,4,positive,Thank you so much for your wonderful review! W...,True,True
4,5,positive,Thank you so much for your wonderful review! W...,True,True



JSON responses: 5 / 5
Schema-valid responses: 5 / 5

TASK 6 COMPLETE


In [17]:
# Task 7: MULTI-TURN CONTEXT

import json

# Customer review used for the conversation

customer_review = df["Review Text"].dropna().iloc[0]


# Turn 1 — Analyze the customer's review

turn_1_prompt = f"""
You are a customer-service assistant for an e-commerce
clothing company.

Analyze the following customer review and remember the
important details because the customer will ask a follow-up
question.

Customer review:
{customer_review}

Return ONLY valid JSON:

{{
  "sentiment": "positive|negative|neutral",
  "summary": "string",
  "important_details": "string"
}}

Do not add any other fields.
Do not use Markdown.
"""


turn_1_response = call_llm_with_retry(
    prompt=turn_1_prompt,
    temperature=0.0,
    max_tokens=200
)


print("\n======================================")
print("PART 3 — TASK 7")
print("MULTI-TURN CONTEXT")
print("======================================")

print("\nTURN 1 — Customer review:")
print(customer_review)

print("\nTURN 1 — Model response:")
print(turn_1_response)


# Turn 2 — Follow-up using the previous context

turn_2_prompt = f"""
You are continuing a customer-service conversation.

The customer's original review was:

{customer_review}

Your previous analysis was:

{turn_1_response}

Now the customer asks:

"Based on my review, what was the main thing I liked or
disliked about the product?"

Use the information from the original review and the previous
analysis to answer the customer's question.

Return ONLY valid JSON:

{{
  "answer": "string",
  "based_on_previous_context": true
}}

Do not add any other fields.
Do not use Markdown.
"""


turn_2_response = call_llm_with_retry(
    prompt=turn_2_prompt,
    temperature=0.0,
    max_tokens=150
)


print("\nTURN 2 — Customer follow-up:")
print("Based on my review, what was the main thing I liked or disliked about the product?")

print("\nTURN 2 — Model response:")
print(turn_2_response)


# Validate both responses

turn_1_valid = False
turn_2_valid = False

try:
    parsed_turn_1 = json.loads(turn_1_response)

    turn_1_valid = (
        set(parsed_turn_1.keys())
        == {
            "sentiment",
            "summary",
            "important_details"
        }
    )

except Exception:
    parsed_turn_1 = None


try:
    parsed_turn_2 = json.loads(turn_2_response)

    turn_2_valid = (
        set(parsed_turn_2.keys())
        == {
            "answer",
            "based_on_previous_context"
        }
        and parsed_turn_2["based_on_previous_context"] is True
    )

except Exception:
    parsed_turn_2 = None


# Final Task 7 result

print("\n======================================")
print("TASK 7 VALIDATION")
print("======================================")

print("Turn 1 valid:", turn_1_valid)
print("Turn 2 valid:", turn_2_valid)

if turn_2_valid:
    print("Multi-turn context successfully demonstrated.")
else:
    print("Multi-turn context validation failed.")

print("\n======================================")
print("TASK 7 COMPLETE")
print("======================================")


PART 3 — TASK 7
MULTI-TURN CONTEXT

TURN 1 — Customer review:
Absolutely wonderful - silky and sexy and comfortable

TURN 1 — Model response:
{
  "sentiment": "positive",
  "summary": "Absolutely wonderful - silky and sexy and comfortable",
  "important_details": "silky, sexy, and comfortable"
}

TURN 2 — Customer follow-up:
Based on my review, what was the main thing I liked or disliked about the product?

TURN 2 — Model response:
{
  "answer": "You liked the product, specifically mentioning its silky, sexy, and comfortable qualities.",
  "based_on_previous_context": true
}

TASK 7 VALIDATION
Turn 1 valid: True
Turn 2 valid: True
Multi-turn context successfully demonstrated.

TASK 7 COMPLETE


In [18]:
# Task 8: API-KEY SECURITY

import os
from dotenv import load_dotenv
from groq import Groq

# Load API key from .env

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY was not found in .env"
    )

print("API key loaded securely from .env.")

# Security checks

# Check that the key exists without displaying the key
print("API key exists:", bool(api_key))

# Confirm that the actual key is never displayed
print("Actual API key value is not displayed.")

# Create Groq client using the securely loaded key

secure_client = Groq(api_key=api_key)

print("Groq client created successfully using the secure key.")

# Task 8 COMPLETE

print("\n======================================")
print("PART 3 — TASK 8")
print("API-KEY SECURITY")
print("======================================")

print("API key stored outside the notebook code: YES")
print("API key displayed in output: NO")
print("API key loaded from .env file: YES")
print("Groq client initialized successfully: YES")

print("\n======================================")
print("TASK 8 COMPLETE")
print("======================================")

API key loaded securely from .env.
API key exists: True
Actual API key value is not displayed.
Groq client created successfully using the secure key.

PART 3 — TASK 8
API-KEY SECURITY
API key stored outside the notebook code: YES
API key displayed in output: NO
API key loaded from .env file: YES
Groq client initialized successfully: YES

TASK 8 COMPLETE


In [19]:
import pandas
import groq

print("pandas:", pandas.__version__)
print("groq:", groq.__version__)

pandas: 2.2.2
groq: 1.6.0
